# GP1 ASR - CRDNN Local Dry-run (macOS / MPS)

Local end-to-end pipeline: build tiny subset -> HP search (random, N=10) -> predict -> `submission.csv`. Smaller trial budget because local runs are slower.

**Assumes:**
- Kaggle dataset already downloaded somewhere on disk.
- `.venv` already exists at `group-projects/gp1/.venv/` (created with `uv venv` + `uv pip install -r requirements.txt`).
- This notebook is opened with the `.venv` kernel selected (VS Code: bottom-right kernel picker; Jupyter: Kernel > Change Kernel).

## 1. User-editable paths

Point `DATA_ROOT` at the directory containing `train.csv`, `train/`, `test.csv`, `test/`.
Default relative layout (if data lives next to the repo): `data/train`, `data/dev`, `data/test`.

In [ ]:
from pathlib import Path

# EDIT ME:
DATA_ROOT = Path("/Users/deniskazhekin/Downloads/asr-2026-spoken-numbers-recognition-challenge")

GP1_ROOT = Path.cwd()
if GP1_ROOT.name != "gp1":
    GP1_ROOT = Path(__file__).resolve().parents[1] if "__file__" in globals() else GP1_ROOT

assert (GP1_ROOT / "scripts" / "train.py").exists(), f"expected gp1 root at {GP1_ROOT}"
assert DATA_ROOT.exists(), f"data root not found: {DATA_ROOT}"
print("GP1 root :", GP1_ROOT)
print("DATA_ROOT:", DATA_ROOT)
for p in sorted(DATA_ROOT.iterdir())[:20]:
    print("  ", p.name, "/" if p.is_dir() else "")

## 2. Device detection

In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print("torch :", torch.__version__)
print("device:", DEVICE)

## 3. Resolve train / dev / test paths

We use the course-provided splits directly:
- `data/train/` — 6 speakers, ~12.5k samples (training)
- `data/dev/`   — 10 speakers, ~2.3k samples (early stopping + HP search)
- `data/test/`  — 14 speakers, ~2.6k samples (Kaggle submission, no labels)

No random 90/10 carving — the course's dev/ is already a proper OOD holdout.

In [ ]:
# Use the real course-provided train/ and dev/ splits.
# - train/ = 6 speakers, 12,553 samples
# - dev/   = 10 speakers, 2,265 samples (OOD for 4 of them)

TRAIN_CSV      = DATA_ROOT / "train" / "train.csv"
TRAIN_WAV_ROOT = DATA_ROOT / "train"
DEV_CSV        = DATA_ROOT / "dev" / "dev.csv"
DEV_WAV_ROOT   = DATA_ROOT / "dev"
TEST_CSV       = DATA_ROOT / "test" / "test.csv"
TEST_WAV_ROOT  = DATA_ROOT / "test"

for p in [TRAIN_CSV, TRAIN_WAV_ROOT, DEV_CSV, DEV_WAV_ROOT, TEST_CSV, TEST_WAV_ROOT]:
    assert p.exists(), f"Missing: {p}"

import csv as _csv
with open(TRAIN_CSV, encoding="utf-8") as fh:
    n_train = sum(1 for _ in _csv.DictReader(fh))
with open(DEV_CSV, encoding="utf-8") as fh:
    n_dev = sum(1 for _ in _csv.DictReader(fh))
print(f"train: {n_train} samples -> {TRAIN_CSV}")
print(f"dev:   {n_dev} samples -> {DEV_CSV}")


## HP Search (Random, N=10)

Random search over a 5-dimensional hyperparameter space following Bergstra & Bengio (2012) JMLR — 10 independent trials (smaller budget for local, where each trial runs on CPU/MPS without GPU acceleration). The formula N >= log(1-p)/log(1-alpha) guarantees p=0.95 probability of finding a configuration in the top alpha fraction of the search space with N trials.

In [ ]:
import math, json, yaml

BASELINE = "crdnn"
CONFIG_NAME = "crdnn.yaml"
N_TRIALS = 10
RANDOM_SEED = 42

OUTPUT_BASE = GP1_ROOT / "_local_runs" / "crdnn"
NUM_WORKERS = 2


def sample_params(rng: random.Random) -> dict:
    """5-dim search space per Bergstra & Bengio (2012)."""
    return {
        "lr": 10 ** rng.uniform(-4, math.log10(5e-3)),
        "dropout": rng.uniform(0.05, 0.25),
        "warmup_steps": rng.randint(500, 8000),
        "grad_clip_norm": rng.choice([0.5, 1.0, 2.0, 5.0]),
        "specaug_time_mask_param": rng.choice([15, 25, 35, 50]),
    }


def patch_config(base_path: Path, params: dict, out_path: Path) -> None:
    """Load base config, merge defaults, apply HP params, write patched YAML."""
    with open(base_path) as f:
        cfg = yaml.safe_load(f)
    if "defaults" in cfg:
        for name in cfg.pop("defaults"):
            with open(base_path.parent / f"{name}.yaml") as f:
                base = yaml.safe_load(f)
            merged = {**base}
            for k, v in cfg.items():
                merged[k] = (
                    {**merged[k], **v}
                    if isinstance(v, dict) and isinstance(merged.get(k), dict)
                    else v
                )
            cfg = merged
    cfg.setdefault("train", {})
    cfg["train"]["lr"] = params["lr"]
    cfg["train"].setdefault("optimizer", {})
    cfg["train"]["optimizer"]["lr"] = params["lr"]
    cfg["train"].setdefault("scheduler", {})
    cfg["train"]["scheduler"]["warmup_steps"] = params["warmup_steps"]
    cfg["train"]["grad_clip_norm"] = params["grad_clip_norm"]
    cfg.setdefault("model", {})
    cfg["model"]["dropout"] = params["dropout"]
    cfg.setdefault("aug", {})
    cfg["aug"]["specaug_time_mask_param"] = params["specaug_time_mask_param"]
    with open(out_path, "w") as f:
        yaml.safe_dump(cfg, f)

In [ ]:
rng = random.Random(RANDOM_SEED)
trials_dir = OUTPUT_BASE / "hp_search"
trials_dir.mkdir(parents=True, exist_ok=True)
trials_log = trials_dir / "trials.jsonl"
best = {"trial_id": None, "cer": float("inf"), "ckpt": None, "params": None}

for trial_id in range(N_TRIALS):
    params = sample_params(rng)
    trial_dir = trials_dir / f"trial_{trial_id:03d}"
    trial_dir.mkdir(parents=True, exist_ok=True)
    patched = trial_dir / "config.yaml"
    patch_config(GP1_ROOT / "configs" / CONFIG_NAME, params, patched)
    cmd = [
        sys.executable, "scripts/train.py",
        "--config", str(patched),
        "--train-csv", str(TRAIN_CSV), "--train-root", str(TRAIN_WAV_ROOT),
        "--dev-csv", str(DEV_CSV), "--dev-root", str(DEV_WAV_ROOT),
        "--output-dir", str(trial_dir),
        "--num-workers", str(NUM_WORKERS),
        "--device", DEVICE,
    ]
    proc = subprocess.run(cmd, cwd=str(GP1_ROOT), capture_output=True, text=True)
    if proc.returncode != 0:
        # Surface the actual error so failed trials are debuggable in-place.
        print(f"\ntrial {trial_id:03d}: FAILED (rc={proc.returncode})")
        print("--- stderr (last 25 lines) ---")
        print("\n".join(proc.stderr.splitlines()[-25:]))
        # Persist the full stderr for later inspection.
        (trial_dir / "stderr.log").write_text(proc.stderr)
        cer = float("inf")
        result = {"error": f"rc={proc.returncode}"}
    else:
        try:
            result = json.loads((trial_dir / "result.json").read_text())
            cer = float(result["best_val_cer"])
        except Exception as exc:
            print(f"\ntrial {trial_id:03d}: could not parse result.json: {exc}")
            cer = float("inf")
            result = {"error": str(exc)}

    with open(trials_log, "a") as f:
        f.write(json.dumps({"trial_id": trial_id, "params": params, "best_val_cer": cer}) + "\n")
    if cer < best["cer"]:
        best = {
            "trial_id": trial_id,
            "cer": cer,
            "ckpt": result.get("best_ckpt_path") if isinstance(result, dict) else None,
            "params": params,
        }
    print(f"trial {trial_id:03d}: CER={cer:.4f}  lr={params['lr']:.2e}")

print(f"\nBEST trial_id={best['trial_id']} CER={best['cer']:.4f}  ckpt={best['ckpt']}")
(trials_dir / "best_trial.json").write_text(json.dumps(best, indent=2, default=str))


## 4. Predict on test data

In [ ]:
# Slice 32 rows from test.csv for a quick inference pass.
TEST_TINY_CSV = SPLIT_DIR / "test_tiny.csv"
with open(TEST_CSV, encoding="utf-8") as fh:
    test_rows = list(csv.DictReader(fh))[:32]
with open(TEST_TINY_CSV, "w", encoding="utf-8", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=list(test_rows[0].keys()))
    w.writeheader()
    w.writerows(test_rows)

SUBMISSION_CSV = trials_dir / "submission.csv"
cmd = [
    sys.executable, "scripts/predict.py",
    "--checkpoint", best["ckpt"],
    "--config", str(trials_dir / f"trial_{best['trial_id']:03d}" / "config.yaml"),
    "--test-csv", str(TEST_TINY_CSV), "--test-root", str(TEST_WAV_ROOT),
    "--output", str(SUBMISSION_CSV),
    "--device", DEVICE,
]
print("$", " ".join(cmd))
subprocess.run(cmd, cwd=str(GP1_ROOT), check=True)
print("Wrote:", SUBMISSION_CSV)

## 5. Preview

In [ ]:
import pandas as pd
df = pd.read_csv(SUBMISSION_CSV)
print(df.shape)
df.head(10)